In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
from diffusers import BitsAndBytesConfig, SD3Transformer2DModel
from diffusers import StableDiffusion3Pipeline
import torch

In [ ]:
# import the models from cache folder
os.environ['HF_HOME'] = "D:/Projetos/hugging-face-cache/"
os.environ["TRANSFORMERS_CACHE"] = "D:/Projetos/hugging-face-cache/"

model_id = "stabilityai/stable-diffusion-3.5-large"

nf4_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
model_nf4 = SD3Transformer2DModel.from_pretrained(
    model_id,
    subfolder="transformer",
    quantization_config=nf4_config,
    torch_dtype=torch.bfloat16
)

pipeline = StableDiffusion3Pipeline.from_pretrained(
    model_id, 
    transformer=model_nf4,
    torch_dtype=torch.bfloat16
)
pipeline.enable_model_cpu_offload()

selected_device = "cuda:0" if torch.cuda.is_available() else "cpu"

print(f'selected_device: {selected_device}')

pipe = pipeline.to(selected_device)

generator = torch.manual_seed(0)

In [ ]:
algorithms_descriptions = {
    "Simulated Annealing 1": "Lava cooling into solid rock, symbolizing search for optimal solution,Highly detailed, 8k, 4k, post processing,Highly detailed, 8k, 4k, post processing.",

    "Local Search 1": "Traveler in rugged landscape, limited to nearby peaks, symbolizing local search,Highly detailed, 8k, 4k, post processing.",
    "Local Search 2": "Person searching for highest point, stuck on lower hills, symbolizing local search,Highly detailed, 8k, 4k, post processing.",

    "Genetic Algorithm 31": "Evolving organisms competing and mutating, finding best solution,Highly detailed, 8k, 4k, post processing.",

    "Particle Swarm Optimization 1": "Particles moving like a flock of birds, converging on best solution,Highly detailed, 8k, 4k, post processing.",
    "Particle Swarm Optimization 2": "Particles moving in harmony, inspired by nature, seeking best outcome,Highly detailed, 8k, 4k, post processing.",

    "Ant Colony Optimization 1": "Ants exploring network, converging on shortest path via pheromones,Highly detailed, 8k, 4k, post processing.",
    "Ant Colony Optimization 2": "Ants finding best path in maze, using pheromone trails as communication,Highly detailed, 8k, 4k, post processing.",

    "Tabu Search 1": "Creature navigating maze, avoiding revisited paths, symbolizing memory-driven search,Highly detailed, 8k, 4k, post processing.",
    "Tabu Search 2": "Intelligent creature avoids dead ends in maze, exploring new promising paths,Highly detailed, 8k, 4k, post processing.",

    "Quantum Annealing 1": "Quantum waves, particle explores all paths, converging on optimal solution,Highly detailed, 8k, 4k, post processing.",
    "Quantum Annealing 2": "Quantum particles tunneling through energy barriers, finding global minimum,Highly detailed, 8k, 4k, post processing.",
    
    "Beam Search 1": "Beams of light scanning forest, focusing on most promising paths,Highly detailed, 8k, 4k, post processing."
}

negative_prompts = {
    "Simulated Annealing": [
        "repetitive patterns, static images, simplistic shapes, abstract visuals, non-dynamic elements, text, person, people",
        "gradual optimization, stagnant phases, repetitive solutions, static representations, energy landscape, text, person, people"
    ],
    "Local Search": [
        "random elements, unstructured visuals, chaotic imagery, disorganized visuals, non-systematic exploration, text, person, people",
        "incremental improvement, local optimum search, chaotic movements, unrealistic solutions, text, person, people"
    ],
    "Genetic Algorithm": [
        "single organisms, non-evolutionary imagery, static visuals, non-dynamic elements, stagnant visuals, text, person, people",
        "crossover, mutation, selection processes, non-interacting entities, text, person, people"
    ],
    "Particle Swarm Optimization": [
        "isolated particles, non-swarming visuals, static imagery, non-collaborative behavior, isolated particles, text, person, people",
        "particle movement, information sharing, still visuals, non-interacting particles, text, person, people"
    ],
    "Ant Colony Optimization": [
        "disconnected trails, non-collaborative visuals, static images, isolated paths, non-interacting ants, text, person, people",
        "pheromone-based pathfinding, adaptive exploration, collective behavior, non-adaptive visuals, text, person, people"
    ],
    "Tabu Search": [
        "unstructured solutions, repetitive visuals, simple representations, non-dynamic search, random search, text, person, people",
        "memory structures, tabu lists, avoidance of cycles, repetitive searching, text, person, people"
    ],
    "Quantum Annealing": [
        "classical optimization imagery, non-quantum visuals, static representations, non-quantum effects, traditional landscapes, text, person, people",
        "superposition, quantum tunneling, solution exploration, linear representations, text, person, people"
    ],
    "Beam Search": [
        "non-heuristic visuals, unstructured searches, chaotic images, overly complex representations, random solution searches, text, person, people",
        "fixed-width beam, promising candidates, unfocused visuals, scattered paths, text, person, people"
    ]
}

params = {
    "Simulated Annealing": {"guidance_scale": [8.5,18.0]},
    "Quantum Annealing": {"guidance_scale": [7.5,8.5]},
    "Local Search": {"guidance_scale": [12.0,18.0]},
    "Particle Swarm Optimization": {"guidance_scale": [7.5,12.0]},
    "Genetic Algorithm": {"guidance_scale": [12.0,18.0]},
    "Ant Colony Optimization": {"guidance_scale": [8.5,12.0]},
    "Beam Search": {"guidance_scale": [18.0]},
    "Tabu Search": {"guidance_scale": [7.5,12.0]}
}

In [ ]:
num_inference_steps = 50
for alg in algorithms_descriptions.keys():
    for neg_index in np.arange(2):
        for guidance_scale in params[alg[:-1].strip()]["guidance_scale"]:
        
            prompt = algorithms_descriptions[alg]
            negative_prompt = negative_prompts[alg[:-2]][neg_index]

            image = pipe(
                prompt,
                generator=generator,
                negative_prompt=negative_prompt,
                guidance_scale=guidance_scale,
                num_inference_steps=num_inference_steps
            ).images[0]

            image.save(
                f"images/stable-diffusion-3.5-large/{alg}_{guidance_scale}_{num_inference_steps}_{neg_index}.png")